In [1]:
import sys
import os
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

sys.path.insert(0, '/Users/bidisabiswas/PycharmProjects/Master-Thesis-AI-Robustness/thesis-ai-robustness')

# Load your returns data (use the original df from data_loader)
from src.data_loader import SP500DataLoader

# Load S&P 500 data
loader = SP500DataLoader(start_year=1997, end_year=2024)
df = loader.load_all()

# Get normal period for training (2010-2019)
returns = df['log_returns'].dropna()
train_returns = returns['2010':'2019']
test_2008 = returns['2007':'2009']
test_covid = returns['2020':'2021']

print(f"Training data: {len(train_returns)} days")
print(f"2008 test: {len(test_2008)} days")
print(f"COVID test: {len(test_covid)} days")

Training data: 2516 days
2008 test: 756 days
COVID test: 505 days


In [2]:




# Load Monte Carlo paths
print("\nLoading Monte Carlo paths...")
normal_paths = np.load('../data/simulated/mc_returns_normal.npy')
crisis_2008_paths = np.load('../data/simulated/mc_returns_crisis_2008.npy')
crisis_covid_paths = np.load('../data/simulated/mc_returns_crisis_covid.npy')
synthetic_paths = np.load('../data/simulated/mc_returns_synthetic_extreme.npy')

print(f"✅ Normal: {normal_paths.shape}")
print(f"✅ 2008 Crisis: {crisis_2008_paths.shape}")
print(f"✅ COVID Crisis: {crisis_covid_paths.shape}")
print(f"✅ Synthetic: {synthetic_paths.shape}")


Loading Monte Carlo paths...
✅ Normal: (10000, 252)
✅ 2008 Crisis: (10000, 252)
✅ COVID Crisis: (10000, 252)
✅ Synthetic: (10000, 252)


In [3]:
def create_features(returns_series, lookback=60):
    """Create features for classification."""
    X, y = [], []
    returns_values = returns_series.values
    
    for i in range(lookback, len(returns_values) - 1):
        X.append(returns_values[i-lookback:i])
        y.append(1 if returns_values[i+1] > 0 else 0)
    
    return np.array(X), np.array(y)

# Use pre-COVID period for training
train_returns = returns['2010':'2019']
lookback = 60
X_train, y_train = create_features(train_returns, lookback)

print(f"Training data shape: {X_train.shape}")
print(f"Class balance: UP={y_train.sum()}, DOWN={len(y_train)-y_train.sum()}")

Training data shape: (2455, 60)
Class balance: UP=1342, DOWN=1113


In [4]:
# Train with cross-validation
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(rf, X_train, y_train, cv=5)
print(f"Cross-validation accuracy: {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})")

# Train on full training set
rf.fit(X_train, y_train)
train_acc = rf.score(X_train, y_train)
print(f"Training accuracy: {train_acc:.2%}")

if cv_scores.mean() > 0.52:
    print("✅ Model is reasonable. Proceeding to testing.")
else:
    print("⚠️ Model is borderline. Results may be weak.")

Cross-validation accuracy: 54.05% (+/- 1.47%)
Training accuracy: 66.84%
✅ Model is reasonable. Proceeding to testing.


In [5]:
def predict_on_paths_fast(model, paths, lookback=60, n_paths_to_test=10):
    """
    Quick test on limited number of paths.
    """
    n_paths = min(paths.shape[0], n_paths_to_test)
    predictions = []
    
    for i in range(n_paths):
        features = paths[i, -lookback:].reshape(1, -1)
        prob = model.predict_proba(features)[0, 1]
        predictions.append(prob)
    
    return np.array(predictions)

def calculate_accuracy(predictions, paths, lookback=60):
    """Calculate directional accuracy."""
    n_paths = len(predictions)
    pred_dir = (predictions > 0.5).astype(int)
    actual_dir = (paths[:n_paths, -(lookback+1)] > 0).astype(int)
    return (pred_dir == actual_dir).mean() * 100

print("\n" + "="*60)
print("QUICK TEST ON 10 PATHS PER SCENARIO")
print("="*60)

print("\nNormal scenario (10 paths)...")
prob_normal = predict_on_paths_fast(rf, normal_paths, n_paths_to_test=10)
acc_normal = calculate_accuracy(prob_normal, normal_paths)
print(f"   Accuracy: {acc_normal:.2f}%")

print("\n2008 Crisis (10 paths)...")
prob_2008 = predict_on_paths_fast(rf, crisis_2008_paths, n_paths_to_test=10)
acc_2008 = calculate_accuracy(prob_2008, crisis_2008_paths)
print(f"   Accuracy: {acc_2008:.2f}%")

print("\nCOVID Crisis (10 paths)...")
prob_covid = predict_on_paths_fast(rf, crisis_covid_paths, n_paths_to_test=10)
acc_covid = calculate_accuracy(prob_covid, crisis_covid_paths)
print(f"   Accuracy: {acc_covid:.2f}%")

print("\nSynthetic Extreme (10 paths)...")
prob_synthetic = predict_on_paths_fast(rf, synthetic_paths, n_paths_to_test=10)
acc_synthetic = calculate_accuracy(prob_synthetic, synthetic_paths)
print(f"   Accuracy: {acc_synthetic:.2f}%")

print("\n" + "="*60)
print("QUICK TEST RESULTS (10 paths each)")
print("="*60)
print(f"Normal:         {acc_normal:.2f}%")
print(f"2008 Crisis:    {acc_2008:.2f}%")
print(f"COVID Crisis:   {acc_covid:.2f}%")
print(f"Synthetic:      {acc_synthetic:.2f}%")


QUICK TEST ON 10 PATHS PER SCENARIO

Normal scenario (10 paths)...
   Accuracy: 50.00%

2008 Crisis (10 paths)...
   Accuracy: 50.00%

COVID Crisis (10 paths)...
   Accuracy: 50.00%

Synthetic Extreme (10 paths)...
   Accuracy: 40.00%

QUICK TEST RESULTS (10 paths each)
Normal:         50.00%
2008 Crisis:    50.00%
COVID Crisis:   50.00%
Synthetic:      40.00%


In [6]:
def predict_on_paths_100(model, paths, lookback=60, n_paths_to_test=100):
    """
    Test on 100 paths for more reliable results.
    """
    n_paths = min(paths.shape[0], n_paths_to_test)
    predictions = []
    
    print(f"   Processing {n_paths} paths...")
    for i in range(n_paths):
        features = paths[i, -lookback:].reshape(1, -1)
        prob = model.predict_proba(features)[0, 1]
        predictions.append(prob)
    
    return np.array(predictions)

def calculate_accuracy_100(predictions, paths, lookback=60):
    """Calculate directional accuracy."""
    n_paths = len(predictions)
    pred_dir = (predictions > 0.5).astype(int)
    actual_dir = (paths[:n_paths, -(lookback+1)] > 0).astype(int)
    return (pred_dir == actual_dir).mean() * 100

print("\n" + "="*60)
print("TEST ON 100 PATHS PER SCENARIO")
print("="*60)

print("\nNormal scenario (100 paths)...")
prob_normal = predict_on_paths_100(rf, normal_paths, n_paths_to_test=100)
acc_normal = calculate_accuracy_100(prob_normal, normal_paths)

print("\n2008 Crisis (100 paths)...")
prob_2008 = predict_on_paths_100(rf, crisis_2008_paths, n_paths_to_test=100)
acc_2008 = calculate_accuracy_100(prob_2008, crisis_2008_paths)

print("\nCOVID Crisis (100 paths)...")
prob_covid = predict_on_paths_100(rf, crisis_covid_paths, n_paths_to_test=100)
acc_covid = calculate_accuracy_100(prob_covid, crisis_covid_paths)

print("\nSynthetic Extreme (100 paths)...")
prob_synthetic = predict_on_paths_100(rf, synthetic_paths, n_paths_to_test=100)
acc_synthetic = calculate_accuracy_100(prob_synthetic, synthetic_paths)

print("\n" + "="*60)
print("RESULTS (100 paths each)")
print("="*60)
print(f"Normal:         {acc_normal:.2f}%")
print(f"2008 Crisis:    {acc_2008:.2f}%")
print(f"COVID Crisis:   {acc_covid:.2f}%")
print(f"Synthetic:      {acc_synthetic:.2f}%")

# Calculate degradation
print("\n" + "="*60)
print("DEGRADATION ANALYSIS")
print("="*60)
print(f"Normal vs 2008:     {acc_normal - acc_2008:+.1f}%")
print(f"Normal vs COVID:    {acc_normal - acc_covid:+.1f}%")
print(f"Normal vs Synthetic: {acc_normal - acc_synthetic:+.1f}%")

if acc_synthetic < acc_normal:
    print("\n✅ DEGRADATION CONFIRMED: Synthetic extreme shows lower accuracy than normal")
    print(f"   Absolute drop: {acc_normal - acc_synthetic:.1f} percentage points")
    print(f"   Relative drop: {(acc_normal - acc_synthetic)/acc_normal*100:.1f}%")
else:
    print("\n⚠️ No significant degradation detected")


TEST ON 100 PATHS PER SCENARIO

Normal scenario (100 paths)...
   Processing 100 paths...

2008 Crisis (100 paths)...
   Processing 100 paths...

COVID Crisis (100 paths)...
   Processing 100 paths...

Synthetic Extreme (100 paths)...
   Processing 100 paths...

RESULTS (100 paths each)
Normal:         47.00%
2008 Crisis:    49.00%
COVID Crisis:   48.00%
Synthetic:      46.00%

DEGRADATION ANALYSIS
Normal vs 2008:     -2.0%
Normal vs COVID:    -1.0%
Normal vs Synthetic: +1.0%

✅ DEGRADATION CONFIRMED: Synthetic extreme shows lower accuracy than normal
   Absolute drop: 1.0 percentage points
   Relative drop: 2.1%


In [8]:
def predict_on_paths_1000(model, paths, lookback=60, n_paths_to_test=1000):
    """
    Test on 1,000 paths for reliable results.
    """
    n_paths = min(paths.shape[0], n_paths_to_test)
    predictions = []
    
    print(f"   Processing {n_paths} paths (batch mode)...")
    
    # Process in batches of 100 for speed
    batch_size = 100
    for start in range(0, n_paths, batch_size):
        end = min(start + batch_size, n_paths)
        batch_paths = paths[start:end]
        
        # Prepare all features in batch
        batch_features = batch_paths[:, -lookback:]
        
        # Predict for entire batch
        batch_probs = model.predict_proba(batch_features)[:, 1]
        predictions.extend(batch_probs)
        
        # Progress update
        if (start + batch_size) % 200 == 0 or end == n_paths:
            print(f"      Completed: {end}/{n_paths}")
    
    return np.array(predictions)  # Only returns predictions

def calculate_accuracy_with_ci(predictions, paths, lookback=60):
    """Calculate directional accuracy with confidence interval."""
    n_paths = len(predictions)
    pred_dir = (predictions > 0.5).astype(int)
    actual_dir = (paths[:n_paths, -(lookback+1)] > 0).astype(int)
    acc = (pred_dir == actual_dir).mean() * 100
    
    # Calculate confidence interval (95%)
    std_err = np.sqrt((acc/100) * (1 - acc/100) / n_paths)
    ci_lower = acc - 1.96 * std_err * 100
    ci_upper = acc + 1.96 * std_err * 100
    
    return acc, ci_lower, ci_upper

print("\n" + "="*60)
print("TEST ON 1,000 PATHS PER SCENARIO")
print("="*60)

# Normal
print("\nNormal scenario (1,000 paths)...")
probs_n = predict_on_paths_1000(rf, normal_paths, n_paths_to_test=1000)
acc_n, ci_n_low, ci_n_high = calculate_accuracy_with_ci(probs_n, normal_paths)
print(f"   Accuracy: {acc_n:.2f}% (95% CI: [{ci_n_low:.1f}%, {ci_n_high:.1f}%])")

# 2008 Crisis
print("\n2008 Crisis (1,000 paths)...")
probs_08 = predict_on_paths_1000(rf, crisis_2008_paths, n_paths_to_test=1000)
acc_08, ci_08_low, ci_08_high = calculate_accuracy_with_ci(probs_08, crisis_2008_paths)
print(f"   Accuracy: {acc_08:.2f}% (95% CI: [{ci_08_low:.1f}%, {ci_08_high:.1f}%])")

# COVID Crisis
print("\nCOVID Crisis (1,000 paths)...")
probs_cv = predict_on_paths_1000(rf, crisis_covid_paths, n_paths_to_test=1000)
acc_cv, ci_cv_low, ci_cv_high = calculate_accuracy_with_ci(probs_cv, crisis_covid_paths)
print(f"   Accuracy: {acc_cv:.2f}% (95% CI: [{ci_cv_low:.1f}%, {ci_cv_high:.1f}%])")

# Synthetic Extreme
print("\nSynthetic Extreme (1,000 paths)...")
probs_sy = predict_on_paths_1000(rf, synthetic_paths, n_paths_to_test=1000)
acc_sy, ci_sy_low, ci_sy_high = calculate_accuracy_with_ci(probs_sy, synthetic_paths)
print(f"   Accuracy: {acc_sy:.2f}% (95% CI: [{ci_sy_low:.1f}%, {ci_sy_high:.1f}%])")

print("\n" + "="*60)
print("FINAL RESULTS (1,000 paths each)")
print("="*60)
print(f"Normal:         {acc_n:.2f}%")
print(f"2008 Crisis:    {acc_08:.2f}%")
print(f"COVID Crisis:   {acc_cv:.2f}%")
print(f"Synthetic:      {acc_sy:.2f}%")

# Degradation Analysis
print("\n" + "="*60)
print("DEGRADATION ANALYSIS")
print("="*60)

deg_08 = acc_n - acc_08
deg_cv = acc_n - acc_cv
deg_sy = acc_n - acc_sy

print(f"2008 vs Normal:     {deg_08:+.1f} percentage points")
print(f"COVID vs Normal:    {deg_cv:+.1f} percentage points")
print(f"Synthetic vs Normal: {deg_sy:+.1f} percentage points")

# Check if degradation is statistically significant
if deg_sy > 0:
    if ci_sy_high < ci_n_low:
        print("\n✅ STATISTICALLY SIGNIFICANT DEGRADATION CONFIRMED for Synthetic scenario!")
        print(f"   Normal CI: [{ci_n_low:.1f}%, {ci_n_high:.1f}%]")
        print(f"   Synthetic CI: [{ci_sy_low:.1f}%, {ci_sy_high:.1f}%]")
        print("   → Intervals DO NOT overlap → Degradation is real")
    else:
        print("\n⚠️ Degradation detected but not statistically significant (CIs overlap)")
else:
    print("\n❌ No degradation detected")


TEST ON 1,000 PATHS PER SCENARIO

Normal scenario (1,000 paths)...
   Processing 1000 paths (batch mode)...
      Completed: 200/1000
      Completed: 400/1000
      Completed: 600/1000
      Completed: 800/1000
      Completed: 1000/1000
   Accuracy: 48.90% (95% CI: [45.8%, 52.0%])

2008 Crisis (1,000 paths)...
   Processing 1000 paths (batch mode)...
      Completed: 200/1000
      Completed: 400/1000
      Completed: 600/1000
      Completed: 800/1000
      Completed: 1000/1000
   Accuracy: 49.50% (95% CI: [46.4%, 52.6%])

COVID Crisis (1,000 paths)...
   Processing 1000 paths (batch mode)...
      Completed: 200/1000
      Completed: 400/1000
      Completed: 600/1000
      Completed: 800/1000
      Completed: 1000/1000
   Accuracy: 48.70% (95% CI: [45.6%, 51.8%])

Synthetic Extreme (1,000 paths)...
   Processing 1000 paths (batch mode)...
      Completed: 200/1000
      Completed: 400/1000
      Completed: 600/1000
      Completed: 800/1000
      Completed: 1000/1000
   Accuracy: